<a href="https://colab.research.google.com/github/ishleensingh/Rule-based-Career-Decision-System/blob/main/career_recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Career Recommendation System Notebook
This notebook implements a hybrid career recommendation system using NLP, rule-based logic, cosine similarity, DAG sequencing, Flask, and MongoDB.

## 1. Import Libraries and Setup Environment
Install and import required Python libraries such as pandas, numpy, sklearn, spaCy, networkx, Flask, pymongo, and json.

In [28]:
# Install required libraries if running in Colab or a fresh environment
# Uncomment the next lines in a new notebook environment if needed.
!pip install pandas numpy scikit-learn spacy nltk networkx flask pymongo
# !python -m spacy download en_core_web_sm

import json
import re
from typing import List, Dict, Any

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

# Removed spacy import as it's no longer used for skill extraction
from nltk.corpus import stopwords

from flask import Flask, request, jsonify
from pymongo import MongoClient

# Removed spaCy model loading as it's no longer used

# Ensure NLTK stopwords are available
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

STOPWORDS = set(stopwords.words('english'))

print("Libraries imported successfully")

Libraries imported successfully


## 2. Load and Inspect Csv Job Dataset
Load job descriptions from csv files, inspect schema and metadata, and extract fields like job title, requirements, and descriptions.

In [29]:
def load_job_data(csv_path: str) -> List[Dict[str, Any]]:
    df = pd.read_csv(csv_path)
    # Assuming the CSV contains the job data in a format convertible to a list of dicts
    return df.to_dict(orient='records')

# The example_json_path and sample_jobs are no longer needed as we are loading from CSV.
# Removing the logic to create jobs_dataset.json

jobs = load_job_data('/content/jobs_dataset_skills_final.csv')
print(f"Loaded {len(jobs)} jobs")
print(jobs[0])

Loaded 1172 jobs
{'job_title': 'Backend Developer', 'company': 'Spatial Front, Inc', 'location': 'Remote', 'is_remote': 'yes', 'role_category': 'backend_developer', 'seniority_level': nan, 'is_aggregator': 'no', 'skills_str': 'python, java, sql, node.js, aws, docker, kubernetes'}


## 3. NLP Preprocessing and Skill Extraction
Clean text, standardize terms, remove redundancy, extract skill phrases from job descriptions, and build a structured skill list.

In [30]:
def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

KNOWN_SKILLS = [
    'python', 'sql', 'excel', 'data visualization', 'machine learning', 'tensorflow',
    'scikit-learn', 'nlp', 'communication', 'data analysis', 'statistics', 'deep learning',
    'pandas', 'numpy', 'docker', 'aws', 'power bi', 'tableau', 'software engineering'
]

standard_skill_map = {normalize_text(skill): skill for skill in KNOWN_SKILLS}


def extract_skills_from_text(text: str, skill_map: Dict[str, str]) -> List[str]:
    # This function is now simplified, as skills_str is assumed to be pre-cleaned
    # It will primarily handle parsing the comma-separated string and canonicalizing.
    if pd.isna(text) or not text.strip():
        return []

    extracted = set()
    for skill_raw in text.split(','):
        normalized_skill = normalize_text(skill_raw.strip())
        if normalized_skill in skill_map:
            extracted.add(skill_map[normalized_skill])
        # Optionally, you could add logic here to include skills not in KNOWN_SKILLS
        # if they are considered valid by the user (e.g., just add normalized_skill)

    return sorted(extracted)


def build_job_skill_rows(jobs_data: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for i, job in enumerate(jobs_data):
        # Use skills_str directly as the source for skill extraction
        skills_from_str = job.get('skills_str', '')
        skills = extract_skills_from_text(skills_from_str, standard_skill_map)

        rows.append({
            'job_id': job.get('job_title', f'job-{i}'), # Use job_title as ID, or generate one if missing
            'title': job.get('job_title'),
            'description': job.get('description', ''), # 'description' not in CSV, will be empty string
            'requirements': job.get('requirements', ''), # 'requirements' not in CSV, will be empty string
            'skills': skills,
            'skill_count': len(skills),
            'industry': job.get('role_category'), # Map 'industry' to 'role_category' from CSV
            'experience_level': job.get('seniority_level') # Map 'experience_level' to 'seniority_level' from CSV
        })
    return pd.DataFrame(rows)

job_skills_df = build_job_skill_rows(jobs)
print(job_skills_df.head())

                                job_id                                title  \
0                    Backend Developer                    Backend Developer   
1  Backend Developer (Streaming Focus)  Backend Developer (Streaming Focus)   
2               Java Backend Developer               Java Backend Developer   
3         Software Developer (backend)         Software Developer (backend)   
4      Software Engineer III - Backend      Software Engineer III - Backend   

  description requirements                      skills  skill_count  \
0                           [aws, docker, python, sql]            4   
1                                        [aws, docker]            2   
2                                                [sql]            1   
3                                             [python]            1   
4                            [aws, excel, python, sql]            4   

            industry experience_level  
0  backend_developer              NaN  
1  backend_develop

## 4. Convert Structured Skills to CSV
Convert extracted skills and job metadata into a pandas DataFrame and save structured output as CSV for downstream processing.

In [31]:
job_skills_df.head()

,job_id,title,description,requirements,skills,skill_count,industry,experience_level
0,Backend Developer,Backend Developer,,,"[aws, docker, python, sql]",4,backend_developer,NaN
1,Backend Developer (Streaming Focus),Backend Developer (Streaming Focus),,,"[aws, docker]",2,backend_developer,NaN
2,Java Backend Developer,Java Backend Developer,,,[sql],1,backend_developer,NaN
3,Software Developer (backend),Software Developer (backend),,,[python],1,backend_developer,NaN
4,Software Engineer III - Backend,Software Engineer III - Backend,,,"[aws, excel, python, sql]",4,backend_developer,NaN


## 5. Build Rule-Based Decision Engine
Implement IF-THEN rules to assess user skills, experience, and preferences, detect missing critical skills, and generate deterministic career guidance.

In [32]:
RULES = [
    {
        'role': 'Machine Learning Engineer',
        'required_skills': {'python', 'machine learning', 'tensorflow', 'scikit-learn'},
        'preferred_skills': {'deep learning', 'statistics', 'software engineering'},
        'recommendation': 'Consider machine learning engineering roles and strengthen model deployment experience.'
    },
    {
        'role': 'Data Analyst',
        'required_skills': {'python', 'sql', 'excel', 'data visualization'},
        'preferred_skills': {'tableau', 'power bi', 'statistics'},
        'recommendation': 'Data analyst is a good match; gain exposure to business reporting and dashboards.'
    }
]


def rule_based_recommendations(user_skills: List[str], user_experience_level: str = None) -> Dict[str, Any]:
    skills_set = set(normalize_text(skill) for skill in user_skills)
    results = []
    for rule in RULES:
        required_missing = [skill for skill in rule['required_skills'] if normalize_text(skill) not in skills_set]
        preferred_missing = [skill for skill in rule['preferred_skills'] if normalize_text(skill) not in skills_set]
        match_score = 1 - len(required_missing) / max(1, len(rule['required_skills']))
        results.append({
            'role': rule['role'],
            'match_score': round(match_score, 2),
            'required_missing': required_missing,
            'preferred_missing': preferred_missing,
            'recommendation': rule['recommendation']
        })
    return {'rules': sorted(results, key=lambda r: (-r['match_score'], len(r['required_missing'])))}

user_profile = {
    'name': 'Alice',
    'skills': ['Python', 'SQL', 'Data Visualization', 'Communication'],
    'experience_level': 'Mid',
    'career_interest': 'data'
}

rule_output = rule_based_recommendations(user_profile['skills'], user_profile['experience_level'])
print(json.dumps(rule_output, indent=2))

{
  "rules": [
    {
      "role": "Data Analyst",
      "match_score": 0.75,
      "required_missing": [
        "excel"
      ],
      "preferred_missing": [
        "statistics",
        "tableau",
        "power bi"
      ],
      "recommendation": "Data analyst is a good match; gain exposure to business reporting and dashboards."
    },
    {
      "role": "Machine Learning Engineer",
      "match_score": 0.25,
      "required_missing": [
        "tensorflow",
        "scikit-learn",
        "machine learning"
      ],
      "preferred_missing": [
        "statistics",
        "software engineering",
        "deep learning"
      ],
      "recommendation": "Consider machine learning engineering roles and strengthen model deployment experience."
    }
  ]
}


## 6. Compute Cosine Similarity for Skill Matching
Vectorize skill sets with CountVectorizer or TF-IDF, compute cosine similarity between user skills and job role skill vectors, and rank career matches.

In [33]:
def build_skill_corpus(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['skill_text'] = df['skills'].apply(lambda skills: ' '.join(normalize_text(s) for s in skills))
    return df

job_skill_corpus = build_skill_corpus(job_skills_df)
vectorizer = CountVectorizer()
job_vectors = vectorizer.fit_transform(job_skill_corpus['skill_text'])


def similarity_rankings(user_skills: List[str], job_df: pd.DataFrame) -> pd.DataFrame:
    user_text = ' '.join(normalize_text(s) for s in user_skills)
    user_vector = vectorizer.transform([user_text])
    similarity_scores = cosine_similarity(user_vector, job_vectors).flatten()
    result = job_df.copy()
    result['similarity_score'] = similarity_scores
    return result.sort_values(by='similarity_score', ascending=False)

similarity_df = similarity_rankings(user_profile['skills'], job_skill_corpus)
print(similarity_df[['job_id', 'title', 'skills', 'similarity_score']].head())

                                                 job_id  \
1089                        Software Developer – Senior   
1093                           Software Developer – Mid   
71                      AI Backend Developer JavaScript   
80                                    Backend Developer   
1068  Software Developer Level 2 with Security Clear...   

                                                  title         skills  \
1089                        Software Developer – Senior  [python, sql]   
1093                           Software Developer – Mid  [python, sql]   
71                      AI Backend Developer JavaScript  [python, sql]   
80                                    Backend Developer  [python, sql]   
1068  Software Developer Level 2 with Security Clear...  [python, sql]   

      similarity_score  
1089               1.0  
1093               1.0  
71                 1.0  
80                 1.0  
1068               1.0  


## 7. Generate DAG-Based Learning Path
Model skill dependencies as a directed acyclic graph, perform topological sorting, and generate an ordered learning path for missing or underdeveloped skills.

In [34]:
skill_dependencies = {
    'machine learning': ['python', 'statistics'],
    'deep learning': ['machine learning', 'tensorflow'],
    'data visualization': ['python', 'excel'],
    'tensorflow': ['python'],
    'scikit-learn': ['python']
}


def build_skill_dag(dependencies: Dict[str, List[str]]) -> nx.DiGraph:
    graph = nx.DiGraph()
    for skill, prereqs in dependencies.items():
        graph.add_node(skill)
        for prereq in prereqs:
            graph.add_edge(prereq, skill)
    if not nx.is_directed_acyclic_graph(graph):
        raise ValueError('Skill dependency graph must be acyclic')
    return graph

skill_graph = build_skill_dag(skill_dependencies)


def generate_learning_path(user_skills: List[str], graph: nx.DiGraph) -> List[str]:
    known = set(normalize_text(skill) for skill in user_skills)
    missing = [node for node in graph.nodes if normalize_text(node) not in known]
    required_prereqs = set()
    for skill in missing:
        for prereq in nx.ancestors(graph, skill):
            if normalize_text(prereq) not in known:
                required_prereqs.add(prereq)
    subgraph = graph.subgraph(required_prereqs.union(missing)).copy()
    path = list(nx.topological_sort(subgraph))
    return [skill for skill in path if normalize_text(skill) not in known]

learning_path = generate_learning_path(user_profile['skills'], skill_graph)
print('Suggested learning path:', learning_path)

Suggested learning path: ['statistics', 'tensorflow', 'excel', 'scikit-learn', 'machine learning', 'deep learning']


## 8. Persist Jobs and Users to MongoDB

This cell is commented out

In [35]:
#MONGO_URI = 'mongodb://localhost:27017/'
#client = MongoClient(MONGO_URI)
#db = client['career_recommendation']
#job_collection = db['jobs']
#user_collection = db['users']

#job_documents = []
#for _, row in job_skills_df.iterrows():
 #   doc = {
  #      'job_id': row['job_id'],
   #     'title': row['title'],
    #    'description': row['description'],
     #   'requirements': row['requirements'],
      #  'skills': row['skills'],
       # 'industry': row['industry'],
        #'experience_level': row['experience_level']
    #}
    #job_documents.append(doc)

#job_collection.delete_many({})
#job_collection.insert_many(job_documents)

#user_doc = {
 #   'name': user_profile['name'],
  #  'skills': user_profile['skills'],
   # 'experience_level': user_profile['experience_level'],
    #'career_interest': user_profile['career_interest'],
    #'rule_recommendations': rule_output,
    #'similarity_recommendations': similarity_df[['job_id', 'title', 'skills', 'similarity_score']].head(5).to_dict(orient='records'),
    #'learning_path': learning_path
#}
#user_collection.delete_many({'name': user_profile['name']})
#user_collection.insert_one(user_doc)

#print('MongoDB persistence complete')
#print('Sample job stored:', job_collection.find_one({'job_id': 'job-1'}))

## 9. Expose Flask Endpoints for Recommendations
Create a Flask app with endpoints for uploading user skills, returning rule-based recommendations, similarity scores, and DAG learning paths.

In [36]:
app = Flask(__name__)

@app.route('/recommend', methods=['POST'])
def recommend():
    payload = request.json or {}
    user_skills = payload.get('skills', [])
    experience_level = payload.get('experience_level')
    career_interest = payload.get('career_interest')

    rules = rule_based_recommendations(user_skills, experience_level)
    similarity_df = similarity_rankings(user_skills, job_skill_corpus)
    similarity_output = similarity_df[['job_id', 'title', 'skills', 'similarity_score']].head(5).to_dict(orient='records')
    learning_path = generate_learning_path(user_skills, skill_graph)

    response = {
        'rule_recommendations': rules,
        'top_role_matches': similarity_output,
        'learning_path': learning_path,
        'input': {
            'skills': user_skills,
            'experience_level': experience_level,
            'career_interest': career_interest
        }
    }
    return jsonify(response)

print('Flask endpoint defined at /recommend')

Flask endpoint defined at /recommend


## 10. Run Flask Application

To interact with the Flask application, you need to run it. In a Colab environment, `app.run()` will block the execution of subsequent cells. For external access or non-blocking execution, you might consider using tools like `ngrok` or running the app in a separate thread/process.

For demonstration purposes, running it here will make the endpoint available for direct calls within the notebook or if exposed externally.

In [37]:
# To run the Flask app, uncomment the line below.
# Be aware that this will block the execution of subsequent cells in Colab.
# If you need to interact with it externally, consider using ngrok.

#app.run(host='0.0.0.0', port=5000, debug=True)
print("Flask app is ready to run. Uncomment 'app.run()' to start the server.")

Flask app is ready to run. Uncomment 'app.run()' to start the server.


## 10. Demonstrate End-to-End Pipeline
Run a sample user input through preprocessing, rule engine, similarity matching, DAG generation, and Flask response formatting to show complete workflow.

In [38]:
sample_input = {
    'skills': ['Python', 'SQL', 'Data Visualization'],
    'experience_level': 'Mid',
    'career_interest': 'analytics'
}

sample_rules = rule_based_recommendations(sample_input['skills'], sample_input['experience_level'])
sample_similarity_df = similarity_rankings(sample_input['skills'], job_skill_corpus)
sample_learning_path = generate_learning_path(sample_input['skills'], skill_graph)

end_to_end_output = {
    'rule_recommendations': sample_rules,
    'top_role_matches': sample_similarity_df[['job_id', 'title', 'skills', 'similarity_score']].head(3).to_dict(orient='records'),
    'learning_path': sample_learning_path
}

print(json.dumps(end_to_end_output, indent=2))

{
  "rule_recommendations": {
    "rules": [
      {
        "role": "Data Analyst",
        "match_score": 0.75,
        "required_missing": [
          "excel"
        ],
        "preferred_missing": [
          "statistics",
          "tableau",
          "power bi"
        ],
        "recommendation": "Data analyst is a good match; gain exposure to business reporting and dashboards."
      },
      {
        "role": "Machine Learning Engineer",
        "match_score": 0.25,
        "required_missing": [
          "tensorflow",
          "scikit-learn",
          "machine learning"
        ],
        "preferred_missing": [
          "statistics",
          "software engineering",
          "deep learning"
        ],
        "recommendation": "Consider machine learning engineering roles and strengthen model deployment experience."
      }
    ]
  },
  "top_role_matches": [
    {
      "job_id": "Software Developer \u2013 Senior",
      "title": "Software Developer \u2013 Senior",
    